# LiH molecular chemistry and tapered VQE

Regenerate the original neutral LiH molecule, discover its diagonal symmetries, and optimize the six-qubit UCCSD ansatz in the Hartree–Fock sector.

## Separate chemistry modeling from the variational quantum kernel

The displayed ansatz factory prepares the tapered Hartree-Fock occupation and
composes ordered Pauli factors for ten excitation operators. It currently uses
explicit FlagQuantum Circuit construction, not a complete native qfunc ansatz.
Host chemistry and symmetry code regenerate the original active-space LiH
Hamiltonian, choose a sector and trim terms; those are not compiler-inferred
chemical assumptions.

Inspect the six-bit sector, 231 retained terms and independent state error
before comparing variational energy. Try another nonzero angle tuple and keep
operator order fixed. The 200-query classical budget and first-order UCCSD
product are bounded approximations, not proof of the exact ground state or
quantum advantage. PySCF is a local scientific prerequisite for regeneration.


Use the optional local PySCF and Matplotlib environments.

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.chemistry.lih_chemistry.overview.variational import (
    PauliOperator,
    append_factor,
)


def ansatz_preview(
    width: int,
    hf_word: int,
    operators: tuple[PauliOperator, ...],
    parameters: tuple[float, ...],
) -> fq.Circuit:
    circuit = fq.Circuit(width, dtype=torch.complex128)
    for wire in range(width):
        if hf_word & (1 << (width - wire - 1)):
            circuit.x(wire)
    for operator, parameter in zip(operators, parameters, strict=True):
        for term in operator.terms:
            append_factor(circuit, term, parameter)
    return circuit


from pathlib import Path

from IPython.display import Image, display

import examples.synthesis.applications.chemistry.lih_chemistry.overview.main as report
import examples.synthesis.applications.chemistry.lih_chemistry.overview.model as model
import examples.synthesis.applications.chemistry.lih_chemistry.overview.symmetry as symmetry
import examples.synthesis.applications.chemistry.lih_chemistry.overview.variational as variational


Use the original neutral singlet LiH geometry and STO-3G basis.

In [ ]:
geometry, basis = model.GEOMETRY, model.BASIS
assert basis == "sto-3g"


Regenerate RHF, MP2, CISD, CCSD, full FCI and frozen-core CASCI.

In [ ]:
molecule = model.regenerate()


Inspect the spatial orbitals and method energies.

In [ ]:
report.print_model(molecule)


Keep five active orbitals and one particle of each spin.

In [ ]:
assert (model.FROZEN, model.ACTIVE_ORBITALS, model.ACTIVE_PARTICLES) == (1, 5, (1, 1))


Inspect the canonical fermionic integral terms.

In [ ]:
print("Fermion term examples:", molecule.fermion.terms[:3], molecule.fermion.terms[-3:])


Map the fresh operator with the native Jordan–Wigner transform.

In [ ]:
sectors = symmetry.analyze(molecule.fermion)
assert len(sectors.hamiltonian.terms) == 276


Retain the one-per-spin Hartree–Fock occupation before tapering.

In [ ]:
unreduced_word = symmetry.HF_WORD
print(f"Interleaved Hartree–Fock occupation: {unreduced_word:010b}")


Discover four diagonal symmetries in the collected Pauli support.

In [ ]:
assert len(sectors.generators) == 4


Check commutation before constructing the reduced frame.

In [ ]:
assert len(sectors.minima) == 16


Read the exact dual X locations of each independent generator.

In [ ]:
assert len(sectors.dual_x) == len(sectors.generators)


Use the paired Pauli generators for the block diagonalizing transform.

In [ ]:
block = sectors.block


Check the transformed generator and its corresponding X operator.

In [ ]:
assert block.is_hermitian()


Inspect the transformed Pauli Hamiltonian.

In [ ]:
print("Transformed term examples:", block.terms[:4])


Compare complete spectra of all sixteen symmetry sectors.

In [ ]:
report.print_sectors(sectors)


Use native Pauli operators for the VQE objective.

In [ ]:
assert sectors.reduction.operator.is_hermitian()


Apply the original 0.001 Ha term filter.

In [ ]:
hamiltonian = symmetry.trim(sectors.reduction.operator, variational.THRESHOLD)
assert len(hamiltonian.terms) == 231


Initialize the six-qubit Hartree–Fock state in the reduced frame.

In [ ]:
print(f"Reduced Hartree–Fock state: {sectors.hf_word:06b}")


Generate the spin-preserving single and double excitation operators.

In [ ]:
operators = variational.excitation_operators(sectors)
assert len(operators) == 10


Use Matplotlib to display the executed VQE trace.

In [ ]:
figures = Path("notebook-figures")


Inspect the first-order native Circuit for the zero initial parameters.

In [ ]:
initial_circuit = ansatz_preview(
    sectors.reduction.n_qubits, sectors.hf_word, operators, (0.0,) * len(operators)
)
print("Initial native UCC circuit operations:", len(initial_circuit))


Run the native optimizer and record all actual objective queries.

In [ ]:
result = variational.fit(sectors)
report.print_optimization(result)


Plot the local energy trace and fresh full-space FCI reference.

In [ ]:
report.save_trace(molecule, result, figures)
display(Image(filename=str(figures / "lih-vqe-trace.png")))
